In [1]:

import duckdb
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = Path('../data/raw')

hospitals = {
    'baylor':                RAW / 'baylor_university_medical_center-69947_parsed.duckdb',
    'methodist':             RAW / 'methodist_dallas_medical_center-6000b_parsed.duckdb',
    'parkland':              RAW / 'parkland_health-6e88d_parsed.duckdb',
    'texas_health_plano':    RAW / 'texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb',
    'medical_city_alliance': RAW / 'medical_city_alliance_hospital-77912_parsed.duckdb',
}

for name, path in hospitals.items():
    print(f"{name:25s} {'OK' if path.exists() else 'MISSING'}  {path.name}")

cons = {name: duckdb.connect(str(path), read_only=True) for name, path in hospitals.items()}

print(f"\nOpened {len(cons)} connections.")

baylor                    OK  baylor_university_medical_center-69947_parsed.duckdb
methodist                 OK  methodist_dallas_medical_center-6000b_parsed.duckdb
parkland                  OK  parkland_health-6e88d_parsed.duckdb
texas_health_plano        OK  texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb
medical_city_alliance     OK  medical_city_alliance_hospital-77912_parsed.duckdb

Opened 5 connections.


In [2]:
con = cons['methodist']

print("=== Tables ===")
print(con.sql("SHOW TABLES").df().to_string(index=False))

print("\n=== standard_charge_details columns (all 44) ===")
cols_details = con.sql("DESCRIBE standard_charge_details").df()
print(cols_details[['column_name', 'column_type']].to_string(index=False))

print("\n=== standard_charges columns ===")
cols_parent = con.sql("DESCRIBE standard_charges").df()
print(cols_parent[['column_name', 'column_type']].to_string(index=False))

=== Tables ===
                   name
       cms_hpt_metadata
              hospitals
modifier_charge_details
       modifier_charges
           mrf_metadata
standard_charge_details
       standard_charges

=== standard_charge_details columns (all 44) ===
               column_name column_type
                 detail_id      BIGINT
                 charge_id      BIGINT
                charge_seq     INTEGER
                 payer_seq     INTEGER
               hospital_id      BIGINT
               description     VARCHAR
              gross_charge      DOUBLE
           discounted_cash      DOUBLE
                   minimum      DOUBLE
                   maximum      DOUBLE
                   setting     VARCHAR
             billing_class     VARCHAR
  additional_generic_notes     VARCHAR
                 drug_unit     VARCHAR
                 drug_type     VARCHAR
                       cpt     VARCHAR
                     hcpcs     VARCHAR
                    ms_drg     VARCHAR
  

In [3]:
SCRATCH = Path('../scratch')
SCRATCH.mkdir(exist_ok=True)

con = cons['methodist']

with open(SCRATCH / 'methodist_schema.txt', 'w') as f:
    f.write("=== Tables ===\n")
    f.write(con.sql("SHOW TABLES").df().to_string(index=False))
    f.write("\n\n=== standard_charge_details columns (all 44) ===\n")
    f.write(con.sql("DESCRIBE standard_charge_details").df()[['column_name','column_type']].to_string(index=False))
    f.write("\n\n=== standard_charges columns ===\n")
    f.write(con.sql("DESCRIBE standard_charges").df()[['column_name','column_type']].to_string(index=False))

print(f"Wrote {SCRATCH / 'methodist_schema.txt'}")

Wrote ..\scratch\methodist_schema.txt


In [4]:
QUERY = """
SELECT
    methodology_normalized,
    COUNT(*) AS n_rows
FROM standard_charge_details
WHERE cpt = '73721'
  AND setting_normalized = 'outpatient'
GROUP BY methodology_normalized
ORDER BY n_rows DESC
"""

results = {}
for name, con in cons.items():
    results[name] = con.sql(QUERY).df()

# Print each hospital's methodology mix
for name, df in results.items():
    total = df['n_rows'].sum()
    print(f"\n=== {name}  (total rate rows: {total}) ===")
    if total == 0:
        print("  (no rows for cpt=73721 setting_normalized=outpatient)")
    else:
        print(df.to_string(index=False))


=== baylor  (total rate rows: 0) ===
  (no rows for cpt=73721 setting_normalized=outpatient)

=== methodist  (total rate rows: 82) ===
         methodology_normalized  n_rows
                   fee schedule      69
                       per diem      11
                      case rate       1
percent of total billed charges       1

=== parkland  (total rate rows: 504) ===
         methodology_normalized  n_rows
percent of total billed charges     174
                   fee schedule     168
                          other     162

=== texas_health_plano  (total rate rows: 80) ===
         methodology_normalized  n_rows
                   fee schedule      56
                          other      14
percent of total billed charges      10

=== medical_city_alliance  (total rate rows: 100) ===
         methodology_normalized  n_rows
                   fee schedule      59
percent of total billed charges      36
                          other       5


In [5]:

con = cons['baylor']

print("=== Q1: Any rows at all for CPT 73721 (any setting)? ===")
q1 = con.sql("""
    SELECT COUNT(*) AS n_rows
    FROM standard_charge_details
    WHERE cpt = '73721'
""").df()
print(q1.to_string(index=False))

print("\n=== Q2: Setting values for CPT 73721 at Baylor ===")
q2 = con.sql("""
    SELECT
        setting,
        setting_normalized,
        COUNT(*) AS n_rows
    FROM standard_charge_details
    WHERE cpt = '73721'
    GROUP BY setting, setting_normalized
    ORDER BY n_rows DESC
""").df()
print(q2.to_string(index=False) if len(q2) else "  (no rows)")

print("\n=== Q3: All distinct setting_normalized values at Baylor (any CPT) ===")
q3 = con.sql("""
    SELECT
        setting_normalized,
        COUNT(*) AS n_rows
    FROM standard_charge_details
    GROUP BY setting_normalized
    ORDER BY n_rows DESC
""").df()
print(q3.to_string(index=False))


=== Q1: Any rows at all for CPT 73721 (any setting)? ===
 n_rows
      0

=== Q2: Setting values for CPT 73721 at Baylor ===
  (no rows)

=== Q3: All distinct setting_normalized values at Baylor (any CPT) ===
setting_normalized  n_rows
        outpatient 1347778
         inpatient  503308


In [6]:


con = cons['baylor']

print("=== Q4: Are CPT 73722 or 73723 (other knee MRI codes) present? ===")
q4 = con.sql("""
    SELECT cpt, COUNT(*) AS n_rows
    FROM standard_charge_details
    WHERE cpt IN ('73721', '73722', '73723')
    GROUP BY cpt
    ORDER BY cpt
""").df()
print(q4.to_string(index=False) if len(q4) else "  (no rows for any of 73721/73722/73723)")

print("\n=== Q5: Is 73721 in the hcpcs column instead? ===")
q5 = con.sql("""
    SELECT hcpcs, COUNT(*) AS n_rows
    FROM standard_charge_details
    WHERE hcpcs IN ('73721', '73722', '73723')
    GROUP BY hcpcs
    ORDER BY hcpcs
""").df()
print(q5.to_string(index=False) if len(q5) else "  (no rows for any of 73721/73722/73723 in hcpcs)")

print("\n=== Q6: Any description containing 'knee' and 'MRI'/'MR'? (sample 10) ===")
q6 = con.sql("""
    SELECT DISTINCT cpt, hcpcs, description
    FROM standard_charge_details
    WHERE LOWER(description) LIKE '%knee%'
      AND (LOWER(description) LIKE '%mri%' OR LOWER(description) LIKE '% mr %' OR LOWER(description) LIKE 'mr %')
    LIMIT 10
""").df()
print(q6.to_string(index=False) if len(q6) else "  (no knee MRI descriptions found)")

=== Q4: Are CPT 73722 or 73723 (other knee MRI codes) present? ===
  (no rows for any of 73721/73722/73723)

=== Q5: Is 73721 in the hcpcs column instead? ===
hcpcs  n_rows
73721     200
73722     200
73723     200

=== Q6: Any description containing 'knee' and 'MRI'/'MR'? (sample 10) ===
  cpt hcpcs                                                                                      description
27369  None INJECTION PROCEDURE FOR CONTRAST KNEE ARTHROGRAPHY OR CONTRAST ENHANCED CT/MRI KNEE ARTHROGRAPHY


In [7]:


con = cons['baylor']

print("=== Q7: Setting breakdown for each hcpcs code ===")
q7 = con.sql("""
    SELECT
        hcpcs,
        setting_normalized,
        COUNT(*) AS n_rows
    FROM standard_charge_details
    WHERE hcpcs IN ('73721', '73722', '73723')
    GROUP BY hcpcs, setting_normalized
    ORDER BY hcpcs, setting_normalized
""").df()
print(q7.to_string(index=False))

print("\n=== Q8: Distinct payer + plan + methodology combos per hcpcs ===")
q8 = con.sql("""
    SELECT
        hcpcs,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT payer_name) AS n_payers,
        COUNT(DISTINCT plan_name) AS n_plans,
        COUNT(DISTINCT methodology_normalized) AS n_methodologies,
        COUNT(DISTINCT (payer_name || '|' || COALESCE(plan_name,'') || '|' || COALESCE(methodology_normalized,''))) AS n_distinct_combos
    FROM standard_charge_details
    WHERE hcpcs IN ('73721', '73722', '73723')
    GROUP BY hcpcs
    ORDER BY hcpcs
""").df()
print(q8.to_string(index=False))

print("\n=== Q9: Sample 5 rows for hcpcs=73721 outpatient at Baylor ===")
q9 = con.sql("""
    SELECT
        hcpcs, setting_normalized, payer_name, plan_name,
        methodology_normalized,
        standard_charge_dollar, standard_charge_percentage, standard_charge_algorithm
    FROM standard_charge_details
    WHERE hcpcs = '73721'
      AND setting_normalized = 'outpatient'
    LIMIT 5
""").df()
print(q9.to_string(index=False))

=== Q7: Setting breakdown for each hcpcs code ===
hcpcs setting_normalized  n_rows
73721          inpatient      20
73721         outpatient     180
73722          inpatient      20
73722         outpatient     180
73723          inpatient      20
73723         outpatient     180

=== Q8: Distinct payer + plan + methodology combos per hcpcs ===
hcpcs  n_rows  n_payers  n_plans  n_methodologies  n_distinct_combos
73721     200        26       30                2                 45
73722     200        26       30                2                 45
73723     200        26       30                2                 45

=== Q9: Sample 5 rows for hcpcs=73721 outpatient at Baylor ===
hcpcs setting_normalized                       payer_name                                                      plan_name          methodology_normalized  standard_charge_dollar  standard_charge_percentage standard_charge_algorithm
73721         outpatient                            Aetna                         

In [8]:


con = cons['baylor']

print("=== Q10: For one specific combo, what varies across its rows? ===")
q10 = con.sql("""
    SELECT
        hcpcs, setting_normalized, payer_name, plan_name, methodology_normalized,
        billing_class_normalized,
        charge_id, charge_seq, payer_seq,
        gross_charge,
        standard_charge_dollar, standard_charge_percentage
    FROM standard_charge_details
    WHERE hcpcs = '73721'
      AND setting_normalized = 'outpatient'
      AND payer_name = 'Aetna'
      AND plan_name = 'Commercial'
    ORDER BY charge_id, charge_seq, payer_seq
""").df()
print(f"Total rows for Aetna/Commercial/73721/outpatient: {len(q10)}")
print(q10.to_string(index=False))

print("\n=== Q11: Distinct combos including billing_class_normalized ===")
q11 = con.sql("""
    SELECT
        hcpcs,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT (payer_name || '|' || COALESCE(plan_name,'') || '|' || COALESCE(methodology_normalized,'') || '|' || COALESCE(billing_class_normalized,''))) AS n_combos_incl_billing_class,
        COUNT(DISTINCT charge_id) AS n_distinct_charge_ids,
        COUNT(DISTINCT description) AS n_distinct_descriptions
    FROM standard_charge_details
    WHERE hcpcs IN ('73721', '73722', '73723')
    GROUP BY hcpcs
    ORDER BY hcpcs
""").df()
print(q11.to_string(index=False))

=== Q10: For one specific combo, what varies across its rows? ===
Total rows for Aetna/Commercial/73721/outpatient: 4
hcpcs setting_normalized payer_name  plan_name          methodology_normalized billing_class_normalized  charge_id  charge_seq  payer_seq  gross_charge  standard_charge_dollar  standard_charge_percentage
73721         outpatient      Aetna Commercial percent of total billed charges                 facility       5304           0          0        3409.3                 1875.12                         NaN
73721         outpatient      Aetna Commercial percent of total billed charges                 facility       5658           0          0        3409.3                 1875.12                         NaN
73721         outpatient      Aetna Commercial percent of total billed charges                 facility      10944           0          0        3409.3                 1875.12                         NaN
73721         outpatient      Aetna Commercial percent of total bi

In [9]:


def query_procedure_rates(con, code: str, setting: str = 'outpatient') -> pd.DataFrame:
    """
    Return rate rows for a given procedure code at one hospital.
    
    Filters:
      - cpt = code OR hcpcs = code  (handles Baylor's hcpcs convention)
      - setting_normalized = setting
    
    Returns columns useful for line-of-business classification and rate analysis.
    Each row is one charge_id × payer × plan × methodology × billing_class combination.
    Call sites must decide whether/how to aggregate across charge_ids.
    """
    q = """
    SELECT
        charge_id,
        description,
        payer_name,
        plan_name,
        payer_group,
        payer_type,
        methodology_normalized,
        billing_class_normalized,
        gross_charge,
        standard_charge_dollar,
        standard_charge_percentage,
        standard_charge_algorithm,
        estimated_amount
    FROM standard_charge_details
    WHERE (cpt = ? OR hcpcs = ?)
      AND setting_normalized = ?
    """
    return con.execute(q, [code, code, setting]).fetchdf()


# Quick sanity check: run it against all 5 hospitals for CPT 73721 outpatient
print("Raw row counts (no dedup, no aggregation):")
raw_counts = {}
for name, con in cons.items():
    df = query_procedure_rates(con, '73721')
    raw_counts[name] = len(df)
    print(f"  {name:25s} {len(df):5d} rows")

Raw row counts (no dedup, no aggregation):
  baylor                      180 rows
  methodist                    82 rows
  parkland                    504 rows
  texas_health_plano           80 rows
  medical_city_alliance       100 rows


In [10]:


print("=== Charge_id fanout per (payer, plan, methodology, billing_class) ===\n")
for name, con in cons.items():
    df = query_procedure_rates(con, '73721')
    if len(df) == 0:
        print(f"{name}: no data\n")
        continue
    
    # Group by the natural "contract" key
    grp = df.groupby(
        ['payer_name', 'plan_name', 'methodology_normalized', 'billing_class_normalized'],
        dropna=False
    )
    
    # For each group: how many charge_ids, and do the dollar rates agree?
    fanout = grp.agg(
        n_rows=('charge_id', 'size'),
        n_distinct_charge_ids=('charge_id', 'nunique'),
        n_distinct_dollar=('standard_charge_dollar', 'nunique'),
        n_distinct_pct=('standard_charge_percentage', 'nunique'),
        dollar_min=('standard_charge_dollar', 'min'),
        dollar_max=('standard_charge_dollar', 'max'),
    ).reset_index()
    
    n_combos = len(fanout)
    n_combos_with_dollar_variation = (fanout['n_distinct_dollar'] > 1).sum()
    n_combos_with_pct_variation = (fanout['n_distinct_pct'] > 1).sum()
    avg_fanout = fanout['n_rows'].mean()
    
    print(f"--- {name} ---")
    print(f"  Total raw rows:                    {len(df)}")
    print(f"  Distinct contract combos:          {n_combos}")
    print(f"  Avg rows per combo (fanout):       {avg_fanout:.2f}")
    print(f"  Combos w/ varying dollar rates:    {n_combos_with_dollar_variation}  ({100*n_combos_with_dollar_variation/n_combos:.0f}%)")
    print(f"  Combos w/ varying pct rates:       {n_combos_with_pct_variation}  ({100*n_combos_with_pct_variation/n_combos:.0f}%)")
    print()

=== Charge_id fanout per (payer, plan, methodology, billing_class) ===

--- baylor ---
  Total raw rows:                    180
  Distinct contract combos:          45
  Avg rows per combo (fanout):       4.00
  Combos w/ varying dollar rates:    0  (0%)
  Combos w/ varying pct rates:       0  (0%)

--- methodist ---
  Total raw rows:                    82
  Distinct contract combos:          82
  Avg rows per combo (fanout):       1.00
  Combos w/ varying dollar rates:    0  (0%)
  Combos w/ varying pct rates:       0  (0%)

--- parkland ---
  Total raw rows:                    504
  Distinct contract combos:          65
  Avg rows per combo (fanout):       7.75
  Combos w/ varying dollar rates:    22  (34%)
  Combos w/ varying pct rates:       0  (0%)

--- texas_health_plano ---
  Total raw rows:                    80
  Distinct contract combos:          40
  Avg rows per combo (fanout):       2.00
  Combos w/ varying dollar rates:    0  (0%)
  Combos w/ varying pct rates:       0  (

In [11]:


def query_procedure_rates_agg(con, code: str, setting: str = 'outpatient') -> pd.DataFrame:
    """
    Aggregated version of query_procedure_rates.
    Returns one row per (payer_name, plan_name, methodology_normalized, billing_class_normalized).
    
    Aggregation:
      - dollar_rate         = median of standard_charge_dollar across charge_ids
      - dollar_min/dollar_max = visibility into combo-level variation (Parkland 34%, THP 9% of combos vary)
      - pct_rate            = max of standard_charge_percentage (stable within combo, max ignores nulls)
      - n_charge_ids        = how many charge_ids contributed to this row
      - gross_min/gross_max = chargemaster gross_charge range across charge_ids
    """
    q = """
    SELECT
        payer_name,
        plan_name,
        payer_group,
        payer_type,
        methodology_normalized,
        billing_class_normalized,
        COUNT(*) AS n_charge_ids,
        MEDIAN(standard_charge_dollar) AS dollar_rate,
        MIN(standard_charge_dollar) AS dollar_min,
        MAX(standard_charge_dollar) AS dollar_max,
        MAX(standard_charge_percentage) AS pct_rate,
        MIN(gross_charge) AS gross_min,
        MAX(gross_charge) AS gross_max,
        STRING_AGG(DISTINCT description, ' | ') AS descriptions
    FROM standard_charge_details
    WHERE (cpt = ? OR hcpcs = ?)
      AND setting_normalized = ?
    GROUP BY payer_name, plan_name, payer_group, payer_type, methodology_normalized, billing_class_normalized
    """
    return con.execute(q, [code, code, setting]).fetchdf()



print("Aggregated row counts (one per payer/plan/methodology/billing_class combo):\n")
for name, con in cons.items():
    df = query_procedure_rates_agg(con, '73721')
    n_combos_with_dollar_variation = (df['dollar_min'] != df['dollar_max']).sum()
    print(f"  {name:25s} {len(df):4d} combos   ({n_combos_with_dollar_variation} with varying dollar across charge_ids)")

Aggregated row counts (one per payer/plan/methodology/billing_class combo):

  baylor                      45 combos   (0 with varying dollar across charge_ids)
  methodist                   82 combos   (0 with varying dollar across charge_ids)
  parkland                    65 combos   (44 with varying dollar across charge_ids)
  texas_health_plano          40 combos   (5 with varying dollar across charge_ids)
  medical_city_alliance       46 combos   (16 with varying dollar across charge_ids)


In [12]:


import re

LOB_PATTERNS = [
    (r'\bCHIP\b',                'medicaid_chip',         'plan_contains_chip'),
    (r'STAR\s*PLUS',             'medicaid_star_plus',    'plan_contains_star_plus'),
    (r'\bSTAR\b',                'medicaid_star',         'plan_contains_star'),
    (r'Medicaid',                'medicaid_other',        'plan_contains_medicaid'),
    (r'Medicare\s*Advantage',    'medicare_advantage',    'plan_contains_medicare_advantage'),
    (r'Exchange|Marketplace|ACA','aca_exchange',          'plan_contains_aca_exchange'),
    (r'Medicare',                'medicare_traditional',  'plan_contains_medicare'),
    (r'Commercial',              'commercial',            'plan_contains_commercial'),
    (r'\bPPO\b|\bHMO\b|\bEPO\b', 'commercial',            'plan_contains_network_type'),
]


def classify_plan(plan_name) -> tuple[str, str]:
    """
    Given a plan_name string, return (lob_bucket, rule_name).
    Returns ('unknown', 'no_match') if no pattern matches.
    """
    if plan_name is None or pd.isna(plan_name) or not str(plan_name).strip():
        return ('unknown', 'null_or_empty_plan_name')
    
    name = str(plan_name)
    for pattern, bucket, rule_name in LOB_PATTERNS:
        if re.search(pattern, name, flags=re.IGNORECASE):
            return (bucket, rule_name)
    return ('unknown', 'no_match')


def add_lob(df: pd.DataFrame) -> pd.DataFrame:
    """Add 'lob' and 'lob_rule' columns based on plan_name."""
    result = df.copy()
    classifications = result['plan_name'].apply(classify_plan)
    result['lob'] = classifications.apply(lambda x: x[0])
    result['lob_rule'] = classifications.apply(lambda x: x[1])
    return result

test_cases = [
    ('ABOVE FPIL AETNA CHIP PERINATE',                      'medicaid_chip'),
    ('AETNA BETTER HEALTH STAR PLUS',                       'medicaid_star_plus'),
    ('AETNA BETTER HEALTH STAR',                            'medicaid_star'),
    ('Texas Medicaid Managed Care',                         'medicaid_other'),
    ('Aetna Medicare Advantage',                            'medicare_advantage'),
    ('Aetna Marketplace HMO',                               'aca_exchange'),
    ('Medicare Part B',                                     'medicare_traditional'),
    ('Aetna Commercial PPO',                                'commercial'),
    ('BCBS PPO',                                            'commercial'),
    ('BSW Plus - Large Group',                              'unknown'),   # this is actually commercial; flagging
    (None,                                                   'unknown'),
    ('',                                                     'unknown'),
    ('Some New Plan We Have Not Seen',                      'unknown'),
]

print("=== Smoke test ===")
all_pass = True
for plan_name, expected in test_cases:
    actual, rule = classify_plan(plan_name)
    status = '✓' if actual == expected else '✗'
    if actual != expected:
        all_pass = False
    plan_display = f"'{plan_name}'" if plan_name else repr(plan_name)
    print(f"  {status} {plan_display:55s} → {actual:25s} (rule: {rule})")

print(f"\n{'ALL PASS' if all_pass else 'FAILURES PRESENT'}")

=== Smoke test ===
  ✓ 'ABOVE FPIL AETNA CHIP PERINATE'                        → medicaid_chip             (rule: plan_contains_chip)
  ✓ 'AETNA BETTER HEALTH STAR PLUS'                         → medicaid_star_plus        (rule: plan_contains_star_plus)
  ✓ 'AETNA BETTER HEALTH STAR'                              → medicaid_star             (rule: plan_contains_star)
  ✓ 'Texas Medicaid Managed Care'                           → medicaid_other            (rule: plan_contains_medicaid)
  ✓ 'Aetna Medicare Advantage'                              → medicare_advantage        (rule: plan_contains_medicare_advantage)
  ✓ 'Aetna Marketplace HMO'                                 → aca_exchange              (rule: plan_contains_aca_exchange)
  ✓ 'Medicare Part B'                                       → medicare_traditional      (rule: plan_contains_medicare)
  ✓ 'Aetna Commercial PPO'                                  → commercial                (rule: plan_contains_commercial)
  ✓ 'BCBS PPO'      

In [13]:

classified = {}
for name, con in cons.items():
    df = query_procedure_rates_agg(con, '73721')
    df = add_lob(df)
    df['hospital'] = name
    classified[name] = df

print("=" * 70)
print("LOB DISTRIBUTION PER HOSPITAL (CPT 73721 outpatient, after aggregation)")
print("=" * 70)
for name, df in classified.items():
    print(f"\n--- {name}  (total combos: {len(df)}) ---")
    counts = df['lob'].value_counts()
    for lob, n in counts.items():
        pct = 100 * n / len(df)
        marker = '  ← REVIEW' if lob == 'unknown' else ''
        print(f"  {lob:25s} {n:3d}  ({pct:4.1f}%){marker}")


print("\n" + "=" * 70)
print("METHODOLOGY MIX WITHIN COMMERCIAL LOB ONLY")
print("=" * 70)
print("(Compare to Cell 3 which conflated all LOBs)\n")
for name, df in classified.items():
    commercial = df[df['lob'] == 'commercial']
    print(f"--- {name}  (commercial combos: {len(commercial)}) ---")
    if len(commercial) == 0:
        print("  (no commercial combos)")
        continue
    mix = commercial['methodology_normalized'].value_counts()
    for method, n in mix.items():
        pct = 100 * n / len(commercial)
        print(f"  {method:35s} {n:3d}  ({pct:4.1f}%)")
    print()

LOB DISTRIBUTION PER HOSPITAL (CPT 73721 outpatient, after aggregation)

--- baylor  (total combos: 45) ---
  unknown                    19  (42.2%)  ← REVIEW
  commercial                 12  (26.7%)
  medicare_advantage          8  (17.8%)
  medicare_traditional        3  ( 6.7%)
  aca_exchange                1  ( 2.2%)
  medicaid_chip               1  ( 2.2%)
  medicaid_other              1  ( 2.2%)

--- methodist  (total combos: 82) ---
  unknown                    46  (56.1%)  ← REVIEW
  medicare_traditional       22  (26.8%)
  commercial                  6  ( 7.3%)
  medicaid_star_plus          3  ( 3.7%)
  aca_exchange                3  ( 3.7%)
  medicare_advantage          1  ( 1.2%)
  medicaid_star               1  ( 1.2%)

--- parkland  (total combos: 65) ---
  medicaid_chip              17  (26.2%)
  unknown                    16  (24.6%)  ← REVIEW
  medicaid_other              9  (13.8%)
  medicaid_star               8  (12.3%)
  medicare_advantage          7  (10.8%)
  medi

In [14]:


SCRATCH = Path('../scratch')
SCRATCH.mkdir(exist_ok=True)


for name, df in classified.items():
    out = SCRATCH / f'day5_{name}_73721_classified.parquet'
    df.to_parquet(out, index=False)
    print(f"Wrote {out}  ({len(df)} combos)")


for name, con in cons.items():
    con.close()
print(f"\nClosed {len(cons)} connections.")

Wrote ..\scratch\day5_baylor_73721_classified.parquet  (45 combos)
Wrote ..\scratch\day5_methodist_73721_classified.parquet  (82 combos)
Wrote ..\scratch\day5_parkland_73721_classified.parquet  (65 combos)
Wrote ..\scratch\day5_texas_health_plano_73721_classified.parquet  (40 combos)
Wrote ..\scratch\day5_medical_city_alliance_73721_classified.parquet  (46 combos)

Closed 5 connections.
